
 # 04 · Trust & Safety Evaluation — YelpConnect

 **Tipo:** Rule Engine / Trust & Safety  
 **Dominio:** Trust & Safety  
 **Entrada:** `workspace.yelp_silver.slv_review_features`  
 **Salida:** `workspace.yelp_silver.slv_review_evaluation`

 ---

 ## Objetivo

 Evaluar las versiones activas de las reglas heurísticas
 Trust & Safety sobre las features previamente calculadas.

 ## Principios

 Este notebook:

 - consume el catálogo gobernado de reglas;
 - ejecuta únicamente reglas activas;
 - conserva la versión exacta de cada regla;
 - registra una ejecución `evaluation_run`;
 - produce una evaluación por Review y versión de regla;
 - genera resultados `SIGNAL`, `NO_SIGNAL` o `NOT_EVALUABLE`;
 - conserva evidencia explicativa de la evaluación.

 Este notebook **NO**:

 - matricula reglas;
 - calcula features;
 - modifica Silver de origen;
 - consolida riesgo por Review;
 - genera una cola de investigación;
 - confirma fraude.

 Un resultado `SIGNAL` representa únicamente una
 **señal heurística**.

 ---

 ## Catálogo gobernado

 - `workspace.yelp_gov.evaluation_rule`
 - `workspace.yelp_gov.evaluation_rule_version`
 - `workspace.yelp_gov.evaluation_run`

 ## Quality Gate previo

 - `workspace.yelp_gov.dq_resultados`

 ## Grano de salida

 **1 Review × 1 Rule Version × 1 Evaluation Run**

 ---

 ## Flujo

 `Silver`

 → `DQ Silver`

 → `Feature Engineering`

 → **Trust & Safety Evaluation**

 → `Risk Consolidation`


In [0]:


# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

import json
import uuid

from datetime import (
    datetime,
    timezone
)

from pyspark.sql import functions as F


# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "yelp_silver"
GOV_SCHEMA = "yelp_gov"

SILVER = (
    f"{CATALOG}."
    f"{SILVER_SCHEMA}"
)

GOV = (
    f"{CATALOG}."
    f"{GOV_SCHEMA}"
)


FEATURE_TABLE = (
    f"{SILVER}.slv_review_features"
)

EVALUATION_TABLE = (
    f"{SILVER}.slv_review_evaluation"
)

RULE_TABLE = (
    f"{GOV}.evaluation_rule"
)

RULE_VERSION_TABLE = (
    f"{GOV}.evaluation_rule_version"
)

RUN_TABLE = (
    f"{GOV}.evaluation_run"
)

DQ_RESULT_TABLE = (
    f"{GOV}.dq_resultados"
)


PIPELINE_NAME = (
    "04_trust_safety_evaluacion"
)

PIPELINE_VERSION = (
    "4.0.0"
)


# ------------------------------------------------------------
# Desarrollo
#
# False = evaluación completa
# True  = limita únicamente el número de Reviews evaluadas.
#
# Las features continúan siendo calculadas sobre el dataset
# completo.
# ------------------------------------------------------------

DEV_MODE = False
DEV_LIMIT = 50000


EVALUATION_RUN_ID = str(
    uuid.uuid4()
)

EVALUATION_STARTED_AT = (
    datetime.now(
        timezone.utc
    )
)


EXPECTED_DQ_SILVER_RULES = 22

SUPPORTED_RULE_IDS = {
    "R001",
    "R002",
    "R003",
    "R004",
    "R005",
    "R006"
}


print("=" * 72)
print("YELPCONNECT - TRUST & SAFETY EVALUATION")
print("=" * 72)

print(
    f"Evaluation Run : "
    f"{EVALUATION_RUN_ID}"
)

print(
    f"Pipeline       : "
    f"{PIPELINE_NAME}"
)

print(
    f"Version        : "
    f"{PIPELINE_VERSION}"
)

print(
    f"DEV MODE       : "
    f"{DEV_MODE}"
)

print(
    f"Input          : "
    f"{FEATURE_TABLE}"
)

print(
    f"Output         : "
    f"{EVALUATION_TABLE}"
)

print("=" * 72)


# COMMAND ----------

# MAGIC %md
# MAGIC # 1 · Precondiciones
# MAGIC
# MAGIC La evaluación Trust & Safety depende de infraestructura
# MAGIC y productos creados previamente.
# MAGIC
# MAGIC Se valida:
# MAGIC
# MAGIC 1. existencia de la tabla de features;
# MAGIC 2. contrato del catálogo de reglas;
# MAGIC 3. contrato del catálogo de versiones;
# MAGIC 4. contrato de `evaluation_run`;
# MAGIC 5. disponibilidad del histórico DQ;
# MAGIC 6. Quality Gate Silver;
# MAGIC 7. consistencia de las versiones activas.
# MAGIC
# MAGIC Este notebook no crea ni corrige infraestructura.


# COMMAND ----------

# ============================================================
# 3. FUNCIÓN DE VALIDACIÓN DE CONTRATOS
# ============================================================

def assert_table_contract(
    table_name,
    expected_contract
):

    if not spark.catalog.tableExists(
        table_name
    ):

        raise RuntimeError(
            f"""
            ❌ TABLA REQUERIDA INEXISTENTE

            Tabla:
            {table_name}
            """
        )


    actual_contract = {

        field.name:
            field.dataType
            .simpleString()
            .lower()

        for field
        in spark.table(
            table_name
        ).schema.fields
    }


    missing_columns = [

        column_name

        for column_name
        in expected_contract

        if column_name
        not in actual_contract
    ]


    if missing_columns:

        raise RuntimeError(
            f"""
            ❌ CONTRATO INCOMPLETO

            Tabla:
            {table_name}

            Columnas faltantes:
            {missing_columns}
            """
        )


    invalid_types = []


    for (
        column_name,
        expected_type
    ) in expected_contract.items():

        actual_type = (
            actual_contract[
                column_name
            ]
        )

        if (
            actual_type
            !=
            expected_type.lower()
        ):

            invalid_types.append(
                {
                    "column":
                        column_name,

                    "expected":
                        expected_type,

                    "actual":
                        actual_type
                }
            )


    if invalid_types:

        raise RuntimeError(
            f"""
            ❌ CONTRATO CON TIPOS INCOMPATIBLES

            Tabla:
            {table_name}

            Detalle:
            {invalid_types}
            """
        )


    print(
        f"✔ Contrato válido: "
        f"{table_name}"
    )


# COMMAND ----------

# ============================================================
# 4. CONTRATOS DE GOBIERNO
# ============================================================

RULE_CONTRACT = {

    "rule_id":
        "string",

    "rule_name":
        "string",

    "rule_type":
        "string",

    "business_definition":
        "string",

    "owner":
        "string",

    "active_flag":
        "boolean",

    "created_at":
        "timestamp"
}


RULE_VERSION_CONTRACT = {

    "rule_version_id":
        "string",

    "rule_id":
        "string",

    "version":
        "string",

    "description":
        "string",

    "parameters_json":
        "string",

    "threshold":
        "double",

    "weight":
        "double",

    "valid_from":
        "timestamp",

    "status":
        "string"
}


RUN_CONTRACT = {

    "evaluation_run_id":
        "string",

    "pipeline_name":
        "string",

    "pipeline_version":
        "string",

    "dataset_version":
        "string",

    "started_at":
        "timestamp",

    "finished_at":
        "timestamp",

    "execution_status":
        "string"
}


DQ_RESULT_CONTRACT = {

    "ejecucion_id":
        "string",

    "ejecucion_ts":
        "timestamp",

    "regla_id":
        "string",

    "capa":
        "string",

    "criticidad":
        "string",

    "paso":
        "boolean"
}


assert_table_contract(
    RULE_TABLE,
    RULE_CONTRACT
)

assert_table_contract(
    RULE_VERSION_TABLE,
    RULE_VERSION_CONTRACT
)

assert_table_contract(
    RUN_TABLE,
    RUN_CONTRACT
)

assert_table_contract(
    DQ_RESULT_TABLE,
    DQ_RESULT_CONTRACT
)


# COMMAND ----------

# ============================================================
# 5. VALIDAR FEATURE TABLE
# ============================================================

if not spark.catalog.tableExists(
    FEATURE_TABLE
):

    raise RuntimeError(
        f"""
        ❌ FEATURE TABLE NO ENCONTRADA

        Tabla:
        {FEATURE_TABLE}

        Ejecutar previamente:
        feature_engineering_ts
        """
    )


REQUIRED_FEATURE_BASE_COLUMNS = {

    "review_id",
    "feature_execution_id"
}


feature_table_columns = set(
    spark.table(
        FEATURE_TABLE
    ).columns
)


missing_base_features = (
    REQUIRED_FEATURE_BASE_COLUMNS
    -
    feature_table_columns
)


if missing_base_features:

    raise RuntimeError(
        f"""
        ❌ CONTRATO DE FEATURES INCOMPLETO

        Columnas faltantes:

        {sorted(missing_base_features)}
        """
    )


print(
    "✔ Feature Table disponible."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 2 · Quality Gate Silver
# MAGIC
# MAGIC Trust & Safety solo puede ejecutarse después de una evaluación
# MAGIC Data Quality Silver válida.
# MAGIC
# MAGIC Se recupera la última ejecución DQ Silver y se verifica:
# MAGIC
# MAGIC - que contenga las 22 reglas esperadas;
# MAGIC - que no tenga fallas de criticidad alta.
# MAGIC
# MAGIC Un `WARNING` no bloquea Trust & Safety.


# COMMAND ----------

# ============================================================
# 6. IDENTIFICAR ÚLTIMA EJECUCIÓN DQ SILVER
# ============================================================

dq_silver = (

    spark.table(
        DQ_RESULT_TABLE
    )

    .filter(
        F.lower(
            F.col(
                "capa"
            )
        )
        ==
        "silver"
    )
)


latest_dq = (

    dq_silver

    .select(
        "ejecucion_id",
        "ejecucion_ts"
    )

    .distinct()

    .orderBy(
        F.desc(
            "ejecucion_ts"
        )
    )

    .first()
)


if latest_dq is None:

    raise RuntimeError(
        """
        ❌ NO EXISTE EJECUCIÓN DQ SILVER

        Trust & Safety requiere ejecutar
        previamente el Quality Gate Silver.
        """
    )


LATEST_DQ_ID = (
    latest_dq.ejecucion_id
)


print(
    f"Última ejecución DQ Silver: "
    f"{LATEST_DQ_ID}"
)


# COMMAND ----------

# ============================================================
# 7. VALIDAR COMPLETITUD DEL DQ RUN
# ============================================================

latest_dq_results = (

    dq_silver

    .filter(
        F.col(
            "ejecucion_id"
        )
        ==
        LATEST_DQ_ID
    )
)


latest_dq_rule_count = (

    latest_dq_results

    .select(
        "regla_id"
    )

    .distinct()

    .count()
)


if (
    latest_dq_rule_count
    !=
    EXPECTED_DQ_SILVER_RULES
):

    raise RuntimeError(
        f"""
        ❌ EJECUCIÓN DQ SILVER INCOMPLETA

        Execution ID:
        {LATEST_DQ_ID}

        Reglas esperadas:
        {EXPECTED_DQ_SILVER_RULES}

        Reglas encontradas:
        {latest_dq_rule_count}
        """
    )


critical_failures = (

    latest_dq_results

    .filter(
        F.lower(
            F.col(
                "criticidad"
            )
        )
        ==
        "alta"
    )

    .filter(
        F.col(
            "paso"
        )
        ==
        False
    )

    .count()
)


if critical_failures > 0:

    raise RuntimeError(
        f"""
        ❌ QUALITY GATE SILVER BLOQUEADO

        Ejecución DQ:
        {LATEST_DQ_ID}

        Fallas críticas:
        {critical_failures}

        Trust & Safety no puede continuar.
        """
    )


print(
    "✔ Quality Gate Silver superado."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 3 · Reglas activas
# MAGIC
# MAGIC Una regla participa únicamente cuando:
# MAGIC
# MAGIC `evaluation_rule.active_flag = true`
# MAGIC
# MAGIC y posee exactamente una versión:
# MAGIC
# MAGIC `evaluation_rule_version.status = 'ACTIVE'`
# MAGIC
# MAGIC Una regla activa sin versión ACTIVE es una inconsistencia de
# MAGIC gobierno y debe detener la ejecución.
# MAGIC
# MAGIC Una regla con múltiples versiones ACTIVE también debe detenerla.
# MAGIC
# MAGIC Las reglas `DRAFT` no participan.


# COMMAND ----------

# ============================================================
# 8. REGLAS FUNCIONALMENTE ACTIVAS
# ============================================================

active_rule_catalog = (

    spark.table(
        RULE_TABLE
    )

    .filter(
        F.col(
            "active_flag"
        )
        ==
        True
    )
)


active_rule_ids = {

    row.rule_id

    for row
    in active_rule_catalog
        .select(
            "rule_id"
        )
        .collect()
}


if not active_rule_ids:

    raise RuntimeError(
        """
        ❌ NO EXISTEN REGLAS TRUST & SAFETY ACTIVAS
        """
    )


print(
    f"✔ Reglas activas en catálogo: "
    f"{len(active_rule_ids)}"
)


# COMMAND ----------

# ============================================================
# 9. VERSIONES ACTIVE
# ============================================================

active_versions = (

    spark.table(
        RULE_VERSION_TABLE
    )

    .filter(
        F.upper(
            F.col(
                "status"
            )
        )
        ==
        "ACTIVE"
    )
)


version_cardinality = (

    active_rule_catalog
    .alias(
        "r"
    )

    .join(

        active_versions
        .alias(
            "v"
        ),

        F.col(
            "r.rule_id"
        )
        ==
        F.col(
            "v.rule_id"
        ),

        "left"
    )

    .groupBy(
        F.col(
            "r.rule_id"
        ).alias(
            "rule_id"
        )
    )

    .agg(
        F.count(
            "v.rule_version_id"
        ).alias(
            "active_version_count"
        )
    )
)


invalid_version_cardinality = (

    version_cardinality

    .filter(
        F.col(
            "active_version_count"
        )
        !=
        1
    )

    .collect()
)


if invalid_version_cardinality:

    raise RuntimeError(
        f"""
        ❌ VERSIONADO TRUST & SAFETY INVÁLIDO

        Cada regla activa debe tener
        exactamente una versión ACTIVE.

        Detalle:

        {invalid_version_cardinality}
        """
    )


print(
    "✔ Cada regla activa posee "
    "exactamente una versión ACTIVE."
)


# COMMAND ----------

# ============================================================
# 10. CONFIGURACIÓN ACTIVA
# ============================================================

active_rules_df = (

    active_rule_catalog
    .alias(
        "r"
    )

    .join(

        active_versions
        .alias(
            "v"
        ),

        F.col(
            "r.rule_id"
        )
        ==
        F.col(
            "v.rule_id"
        ),

        "inner"
    )

    .select(

        F.col(
            "r.rule_id"
        ),

        F.col(
            "r.rule_name"
        ),

        F.col(
            "r.rule_type"
        ),

        F.col(
            "r.business_definition"
        ),

        F.col(
            "r.owner"
        ),

        F.col(
            "v.rule_version_id"
        ),

        F.col(
            "v.version"
        ),

        F.col(
            "v.parameters_json"
        ),

        F.col(
            "v.threshold"
        ),

        F.col(
            "v.weight"
        )
    )
)


active_rule_rows = (
    active_rules_df.collect()
)


# COMMAND ----------

# ============================================================
# 11. REGLAS SOPORTADAS POR EL MOTOR
# ============================================================

unsupported_rules = (
    active_rule_ids
    -
    SUPPORTED_RULE_IDS
)


if unsupported_rules:

    raise RuntimeError(
        f"""
        ❌ EXISTEN REGLAS ACTIVE
        NO IMPLEMENTADAS POR EL MOTOR

        Reglas:
        {sorted(unsupported_rules)}

        Actualizar primero:
        04_trust_safety_evaluacion
        """
    )


print(
    "✔ Todas las reglas ACTIVE "
    "son soportadas por el motor."
)


# COMMAND ----------

# ============================================================
# 12. CONVERTIR CATÁLOGO A CONFIGURACIÓN
# ============================================================

rule_configs = {}


for row in active_rule_rows:

    parameters_raw = (
        row["parameters_json"]
        or
        "{}"
    )


    try:

        parameters = (
            json.loads(
                parameters_raw
            )
        )

    except Exception as exc:

        raise RuntimeError(
            f"""
            ❌ parameters_json INVÁLIDO

            Rule:
            {row['rule_id']}

            JSON:
            {parameters_raw}
            """
        ) from exc


    rule_configs[
        row["rule_id"]
    ] = {

        "rule_version_id":
            row[
                "rule_version_id"
            ],

        "version":
            row[
                "version"
            ],

        "threshold":
            (
                float(
                    row[
                        "threshold"
                    ]
                )

                if row[
                    "threshold"
                ] is not None

                else None
            ),

        "weight":
            (
                float(
                    row[
                        "weight"
                    ]
                )

                if row[
                    "weight"
                ] is not None

                else 0.0
            ),

        "parameters":
            parameters
    }


display(
    active_rules_df
    .orderBy(
        "rule_id"
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC ## Validación de pesos
# MAGIC
# MAGIC Los pesos de las reglas ACTIVE deben sumar exactamente `1.0`
# MAGIC dentro de una tolerancia técnica.
# MAGIC
# MAGIC Esta condición es obligatoria porque `Risk Consolidation`
# MAGIC utilizará posteriormente estos pesos para construir el
# MAGIC `weighted_risk_score`.
# MAGIC
# MAGIC El score resultante será una medida heurística ponderada,
# MAGIC **no una probabilidad estadística de fraude**.


# COMMAND ----------

# ============================================================
# 13. VALIDAR PESOS
# ============================================================

active_weight = sum(

    config[
        "weight"
    ]

    for config
    in rule_configs.values()
)


print(
    f"Peso total ACTIVE: "
    f"{active_weight:.4f}"
)


if abs(
    active_weight
    -
    1.0
) > 0.0001:

    raise RuntimeError(
        f"""
        ❌ PESOS TRUST & SAFETY INVÁLIDOS

        Suma ACTIVE:
        {active_weight}

        Esperado:
        1.0
        """
    )


print(
    "✔ Pesos ACTIVE suman 1.0."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 4 · Features requeridas
# MAGIC
# MAGIC Cada regla depende de un conjunto explícito de features.
# MAGIC
# MAGIC Antes de ejecutar el motor se verifica que todas las features
# MAGIC requeridas por las reglas ACTIVE estén disponibles.
# MAGIC
# MAGIC Si una nueva regla pasa a `ACTIVE` sin que Feature Engineering
# MAGIC soporte sus variables, la ejecución debe fallar en lugar de
# MAGIC ignorarla silenciosamente.


# COMMAND ----------

# ============================================================
# 14. FEATURE REQUIREMENTS
# ============================================================

required_features_by_rule = {

    "R001": {
        "user_reviews_30m"
    },

    "R002": {
        "rating_deviation",
        "business_stars"
    },

    "R003": {
        "text_length",
        "text_repeat_count"
    },

    "R004": {
        "account_age_days",
        "user_review_count",
        "yelping_since"
    },

    "R005": {
        "user_business_review_count",
        "user_business_concentration"
    },

    "R006": {
        "business_reviews_day",
        "business_avg_daily_reviews",
        "business_spike_ratio"
    }
}


feature_table_columns = set(

    spark.table(
        FEATURE_TABLE
    ).columns
)


missing_features = []


for rule_id in sorted(
    active_rule_ids
):

    required_columns = (
        required_features_by_rule[
            rule_id
        ]
    )

    missing = (
        required_columns
        -
        feature_table_columns
    )


    for column_name in sorted(
        missing
    ):

        missing_features.append(
            (
                rule_id,
                column_name
            )
        )


if missing_features:

    raise RuntimeError(
        f"""
        ❌ FALTAN FEATURES REQUERIDAS

        {missing_features}

        Ejecutar o actualizar:
        feature_engineering_ts
        """
    )


print(
    "✔ Features requeridas disponibles."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 5 · Dataset Version
# MAGIC
# MAGIC Cada Evaluation Run debe poder rastrearse hacia una única
# MAGIC ejecución de Feature Engineering.
# MAGIC
# MAGIC `dataset_version` utiliza:
# MAGIC
# MAGIC `feature_execution_id`
# MAGIC
# MAGIC La tabla de features debe contener exactamente un
# MAGIC `feature_execution_id`.
# MAGIC
# MAGIC Si existen cero o múltiples valores, la evaluación se detiene
# MAGIC porque no sería posible garantizar reproducibilidad.


# COMMAND ----------

# ============================================================
# 15. LEER FEATURES
# ============================================================

features = (

    spark.table(
        FEATURE_TABLE
    )
)


if DEV_MODE:

    features = (

        features

        .orderBy(
            "review_id"
        )

        .limit(
            DEV_LIMIT
        )
    )


review_count = (
    features.count()
)


if review_count == 0:

    raise RuntimeError(
        """
        ❌ NO EXISTEN REVIEWS PARA EVALUAR
        """
    )


print(
    f"Reviews a evaluar: "
    f"{review_count:,}"
)


# COMMAND ----------

# ============================================================
# 16. VALIDAR FEATURE EXECUTION
# ============================================================

feature_version_rows = (

    features

    .select(
        "feature_execution_id"
    )

    .where(
        F.col(
            "feature_execution_id"
        ).isNotNull()
    )

    .distinct()

    .limit(
        2
    )

    .collect()
)


if (
    len(
        feature_version_rows
    )
    !=
    1
):

    raise RuntimeError(
        f"""
        ❌ DATASET VERSION AMBIGUA

        Se esperaba exactamente un
        feature_execution_id.

        Valores encontrados:
        {len(feature_version_rows)}

        Reconstruir:
        {FEATURE_TABLE}
        """
    )


DATASET_VERSION = (
    feature_version_rows[
        0
    ]
    .feature_execution_id
)


print(
    f"Feature Version : "
    f"{DATASET_VERSION}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 6 · Evaluation Run
# MAGIC
# MAGIC Antes de ejecutar las reglas se registra una ejecución en:
# MAGIC
# MAGIC `workspace.yelp_gov.evaluation_run`
# MAGIC
# MAGIC El estado inicial es:
# MAGIC
# MAGIC `RUNNING`
# MAGIC
# MAGIC Al finalizar:
# MAGIC
# MAGIC - `SUCCESS` si la evaluación concluye correctamente;
# MAGIC - `FAILED` si ocurre un error después de registrar el Run.
# MAGIC
# MAGIC La tabla `evaluation_run` es infraestructura de gobierno y
# MAGIC **no se crea en este notebook**.


# COMMAND ----------

# ============================================================
# 17. REGISTRAR RUN
# ============================================================

run_schema = """
evaluation_run_id STRING,
pipeline_name STRING,
pipeline_version STRING,
dataset_version STRING,
started_at TIMESTAMP,
finished_at TIMESTAMP,
execution_status STRING
"""


run_df = spark.createDataFrame(

    [
        (
            EVALUATION_RUN_ID,
            PIPELINE_NAME,
            PIPELINE_VERSION,
            DATASET_VERSION,
            EVALUATION_STARTED_AT,
            None,
            "RUNNING"
        )
    ],

    run_schema
)


(
    run_df

    .write

    .format(
        "delta"
    )

    .mode(
        "append"
    )

    .saveAsTable(
        RUN_TABLE
    )
)


print(
    "✔ Evaluation Run registrado."
)


# COMMAND ----------

# ============================================================
# 18. ACTUALIZAR ESTADO DEL RUN
# ============================================================

def update_run_status(
    status
):

    if status not in {
        "SUCCESS",
        "FAILED"
    }:

        raise ValueError(
            f"Estado inválido: {status}"
        )


    spark.sql(
        f"""
        UPDATE
            {RUN_TABLE}

        SET
            execution_status = '{status}',
            finished_at = current_timestamp()

        WHERE
            evaluation_run_id = '{EVALUATION_RUN_ID}'
        """
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # 7 · Motor Trust & Safety
# MAGIC
# MAGIC La configuración funcional proviene del catálogo:
# MAGIC
# MAGIC - versión;
# MAGIC - parámetros;
# MAGIC - threshold;
# MAGIC - peso.
# MAGIC
# MAGIC La implementación técnica de las reglas R001–R006 reside
# MAGIC en este motor.
# MAGIC
# MAGIC Cada regla produce:
# MAGIC
# MAGIC - `SIGNAL`;
# MAGIC - `NO_SIGNAL`;
# MAGIC - `NOT_EVALUABLE`.
# MAGIC
# MAGIC ## NOT_EVALUABLE
# MAGIC
# MAGIC Se utiliza cuando no existen las features necesarias para
# MAGIC evaluar una Review concreta.
# MAGIC
# MAGIC `NOT_EVALUABLE` no equivale a:
# MAGIC
# MAGIC - señal;
# MAGIC - ausencia de riesgo;
# MAGIC - fraude.
# MAGIC
# MAGIC Es una limitación de evaluación.


# COMMAND ----------

# ============================================================
# 19. CONSTRUCTOR DE REGLAS
# ============================================================

def build_rule_struct(
    rule_id
):

    config = (
        rule_configs[
            rule_id
        ]
    )

    params = (
        config[
            "parameters"
        ]
    )

    weight = (
        config[
            "weight"
        ]
    )

    rule_version_id = (
        config[
            "rule_version_id"
        ]
    )


    # ========================================================
    # R001 · Burst Activity
    # ========================================================

    if rule_id == "R001":

        window_minutes = int(

            params.get(
                "window_minutes",
                30
            )
        )


        if window_minutes != 30:

            raise RuntimeError(
                f"""
                ❌ R001 solicita ventana de
                {window_minutes} minutos.

                Feature disponible:
                user_reviews_30m

                Actualizar Feature Engineering.
                """
            )


        min_reviews = int(

            params.get(
                "min_reviews",
                config[
                    "threshold"
                ]
            )
        )


        evaluable = (

            F.col(
                "user_reviews_30m"
            ).isNotNull()
        )


        signal = (

            F.col(
                "user_reviews_30m"
            )
            >=
            min_reviews
        )


        reason = F.concat(

            F.lit(
                "Reviews usuario en 30m: "
            ),

            F.col(
                "user_reviews_30m"
            ).cast(
                "string"
            ),

            F.lit(
                f" | threshold: {min_reviews}"
            )
        )


    # ========================================================
    # R002 · Rating Deviation
    # ========================================================

    elif rule_id == "R002":

        rating_difference = float(

            params.get(
                "rating_difference",
                config[
                    "threshold"
                ]
            )
        )


        evaluable = (

            F.col(
                "business_stars"
            ).isNotNull()

            &

            F.col(
                "rating_deviation"
            ).isNotNull()
        )


        signal = (

            F.col(
                "rating_deviation"
            )
            >=
            rating_difference
        )


        reason = F.concat(

            F.lit(
                "Desviación rating: "
            ),

            F.round(
                F.col(
                    "rating_deviation"
                ),
                2
            ).cast(
                "string"
            ),

            F.lit(
                f" | threshold: "
                f"{rating_difference}"
            )
        )


    # ========================================================
    # R003 · Repeated Text
    # ========================================================

    elif rule_id == "R003":

        min_repetitions = int(

            params.get(
                "min_repetitions",
                config[
                    "threshold"
                ]
            )
        )


        min_text_length = int(

            params.get(
                "min_text_length",
                40
            )
        )


        evaluable = (

            F.col(
                "text_length"
            ).isNotNull()

            &

            F.col(
                "text_repeat_count"
            ).isNotNull()
        )


        signal = (

            (
                F.col(
                    "text_length"
                )
                >=
                min_text_length
            )

            &

            (
                F.col(
                    "text_repeat_count"
                )
                >=
                min_repetitions
            )
        )


        reason = F.concat(

            F.lit(
                "Texto repetido: "
            ),

            F.col(
                "text_repeat_count"
            ).cast(
                "string"
            ),

            F.lit(
                f" veces | longitud mínima: "
                f"{min_text_length}"
            ),

            F.lit(
                f" | repeticiones mínimas: "
                f"{min_repetitions}"
            )
        )


    # ========================================================
    # R004 · New Account High Activity
    # ========================================================

    elif rule_id == "R004":

        max_account_age_days = int(

            params.get(
                "account_age_days",
                30
            )
        )


        min_reviews = int(

            params.get(
                "min_reviews",
                config[
                    "threshold"
                ]
            )
        )


        evaluable = (

            F.col(
                "yelping_since"
            ).isNotNull()

            &

            F.col(
                "account_age_days"
            ).isNotNull()

            &

            F.col(
                "user_review_count"
            ).isNotNull()
        )


        signal = (

            F.col(
                "account_age_days"
            ).between(
                0,
                max_account_age_days
            )

            &

            (
                F.col(
                    "user_review_count"
                )
                >=
                min_reviews
            )
        )


        reason = F.concat(

            F.lit(
                "Edad cuenta: "
            ),

            F.col(
                "account_age_days"
            ).cast(
                "string"
            ),

            F.lit(
                " días | Reviews usuario: "
            ),

            F.col(
                "user_review_count"
            ).cast(
                "string"
            ),

            F.lit(
                f" | max edad: "
                f"{max_account_age_days}"
            ),

            F.lit(
                f" | min reviews: "
                f"{min_reviews}"
            )
        )


    # ========================================================
    # R005 · User-Business Concentration
    # ========================================================

    elif rule_id == "R005":

        concentration = float(

            params.get(
                "concentration",
                config[
                    "threshold"
                ]
            )
        )


        min_business_reviews = int(

            params.get(
                "min_business_reviews",
                3
            )
        )


        evaluable = (

            F.col(
                "user_business_review_count"
            ).isNotNull()

            &

            F.col(
                "user_business_concentration"
            ).isNotNull()
        )


        signal = (

            (
                F.col(
                    "user_business_review_count"
                )
                >=
                min_business_reviews
            )

            &

            (
                F.col(
                    "user_business_concentration"
                )
                >=
                concentration
            )
        )


        reason = F.concat(

            F.lit(
                "Reviews User-Business: "
            ),

            F.col(
                "user_business_review_count"
            ).cast(
                "string"
            ),

            F.lit(
                " | concentración: "
            ),

            F.round(
                F.col(
                    "user_business_concentration"
                ),
                4
            ).cast(
                "string"
            ),

            F.lit(
                f" | threshold: "
                f"{concentration}"
            )
        )


    # ========================================================
    # R006 · Business Review Spike
    # ========================================================

    elif rule_id == "R006":

        min_daily_reviews = int(

            params.get(
                "min_daily_reviews",
                10
            )
        )


        spike_ratio = float(

            params.get(
                "spike_ratio",
                config[
                    "threshold"
                ]
            )
        )


        evaluable = (

            F.col(
                "business_reviews_day"
            ).isNotNull()

            &

            F.col(
                "business_avg_daily_reviews"
            ).isNotNull()

            &

            F.col(
                "business_spike_ratio"
            ).isNotNull()
        )


        signal = (

            (
                F.col(
                    "business_reviews_day"
                )
                >=
                min_daily_reviews
            )

            &

            (
                F.col(
                    "business_spike_ratio"
                )
                >=
                spike_ratio
            )
        )


        reason = F.concat(

            F.lit(
                "Reviews día: "
            ),

            F.col(
                "business_reviews_day"
            ).cast(
                "string"
            ),

            F.lit(
                " | spike ratio: "
            ),

            F.round(
                F.col(
                    "business_spike_ratio"
                ),
                2
            ).cast(
                "string"
            ),

            F.lit(
                f" | min reviews: "
                f"{min_daily_reviews}"
            ),

            F.lit(
                f" | min spike: "
                f"{spike_ratio}"
            )
        )


    else:

        raise RuntimeError(
            f"""
            ❌ REGLA NO IMPLEMENTADA

            {rule_id}
            """
        )


    # ========================================================
    # RESULTADO ESTÁNDAR
    # ========================================================

    evaluation_result = (

        F.when(
            ~evaluable,

            F.lit(
                "NOT_EVALUABLE"
            )
        )

        .when(
            signal,

            F.lit(
                "SIGNAL"
            )
        )

        .otherwise(
            F.lit(
                "NO_SIGNAL"
            )
        )
    )


    # --------------------------------------------------------
    # IMPORTANTE:
    #
    # risk_score representa aquí el aporte de peso de la regla
    # cuando existe SIGNAL.
    #
    # NO es una probabilidad de fraude.
    # --------------------------------------------------------

    risk_score = (

        F.when(
            evaluable
            &
            signal,

            F.lit(
                weight
            )
        )

        .otherwise(
            F.lit(
                0.0
            )
        )
    )


    final_reason = (

        F.when(
            ~evaluable,

            F.lit(
                "No evaluable: features requeridas "
                "no disponibles."
            )
        )

        .otherwise(
            reason
        )
    )


    return F.struct(

        F.lit(
            rule_id
        ).alias(
            "rule_id"
        ),

        F.lit(
            rule_version_id
        ).alias(
            "rule_version_id"
        ),

        evaluation_result.alias(
            "evaluation_result"
        ),

        risk_score.alias(
            "risk_score"
        ),

        final_reason.alias(
            "reason"
        )
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # 8 · Ejecución de reglas
# MAGIC
# MAGIC Las reglas activas se construyen dinámicamente a partir del
# MAGIC catálogo.
# MAGIC
# MAGIC No existe un multiplicador fijo como `Reviews × 6`.
# MAGIC
# MAGIC El número esperado de evaluaciones se calcula como:
# MAGIC
# MAGIC `Reviews × cantidad de Rule Versions ACTIVE`
# MAGIC
# MAGIC De esta forma el grano permanece gobernado por el catálogo.


# COMMAND ----------

# ============================================================
# 20. CONSTRUIR EXPRESIONES
# ============================================================

rule_structs = [

    build_rule_struct(
        rule_id
    )

    for rule_id
    in sorted(
        active_rule_ids
    )
]


ACTIVE_RULE_COUNT = len(
    rule_structs
)


print(
    f"Reglas ejecutadas: "
    f"{ACTIVE_RULE_COUNT}"
)


# COMMAND ----------

# ============================================================
# 21. EVALUACIÓN
# ============================================================

evaluations = (

    features

    .select(

        "review_id",

        F.array(
            *rule_structs
        ).alias(
            "_rule_evaluations"
        )
    )

    .select(

        "review_id",

        F.explode(
            "_rule_evaluations"
        ).alias(
            "rule"
        )
    )

    .select(

        "review_id",

        F.col(
            "rule.rule_id"
        ).alias(
            "rule_id"
        ),

        F.col(
            "rule.rule_version_id"
        ).alias(
            "rule_version_id"
        ),

        F.col(
            "rule.evaluation_result"
        ).alias(
            "evaluation_result"
        ),

        F.col(
            "rule.risk_score"
        ).alias(
            "risk_score"
        ),

        F.col(
            "rule.reason"
        ).alias(
            "reason"
        )
    )

    .withColumn(
        "evaluation_run_id",

        F.lit(
            EVALUATION_RUN_ID
        )
    )

    .withColumn(
        "evaluation_id",

        F.sha2(

            F.concat_ws(

                "||",

                F.col(
                    "review_id"
                ),

                F.col(
                    "rule_version_id"
                ),

                F.col(
                    "evaluation_run_id"
                )
            ),

            256
        )
    )

    .withColumn(
        "evaluated_at",

        F.current_timestamp()
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 9 · Validación del grano
# MAGIC
# MAGIC La cantidad esperada es:
# MAGIC
# MAGIC `Reviews × reglas ACTIVE`
# MAGIC
# MAGIC Además debe cumplirse que cada combinación:
# MAGIC
# MAGIC `review_id + rule_version_id + evaluation_run_id`
# MAGIC
# MAGIC sea única.


# COMMAND ----------

# ============================================================
# 22. VALIDAR GRANO
# ============================================================

EXPECTED_COUNT = (
    review_count
    *
    ACTIVE_RULE_COUNT
)


try:

    ACTUAL_COUNT = (
        evaluations.count()
    )


    print(
        f"Reviews            : "
        f"{review_count:,}"
    )

    print(
        f"Reglas ACTIVE      : "
        f"{ACTIVE_RULE_COUNT}"
    )

    print(
        f"Esperadas          : "
        f"{EXPECTED_COUNT:,}"
    )

    print(
        f"Generadas          : "
        f"{ACTUAL_COUNT:,}"
    )


    if (
        ACTUAL_COUNT
        !=
        EXPECTED_COUNT
    ):

        raise RuntimeError(
            f"""
            ❌ GRANO DE EVALUACIÓN INCORRECTO

            Reviews:
            {review_count}

            Reglas:
            {ACTIVE_RULE_COUNT}

            Esperadas:
            {EXPECTED_COUNT}

            Generadas:
            {ACTUAL_COUNT}
            """
        )


    duplicate_evaluations = (

        evaluations

        .groupBy(
            "review_id",
            "rule_version_id",
            "evaluation_run_id"
        )

        .count()

        .filter(
            F.col(
                "count"
            )
            !=
            1
        )

        .count()
    )


    if duplicate_evaluations > 0:

        raise RuntimeError(
            f"""
            ❌ EVALUACIONES DUPLICADAS

            Combinaciones duplicadas:
            {duplicate_evaluations}
            """
        )


    print(
        "✔ Grano Review × Rule Version × Run validado."
    )


    # ========================================================
    # 23. VALIDAR CONTRATO DE TABLA EXISTENTE
    # ========================================================

    EVALUATION_CONTRACT = {

        "review_id":
            "string",

        "rule_id":
            "string",

        "rule_version_id":
            "string",

        "evaluation_result":
            "string",

        "risk_score":
            "double",

        "reason":
            "string",

        "evaluation_run_id":
            "string",

        "evaluation_id":
            "string",

        "evaluated_at":
            "timestamp"
    }


    if spark.catalog.tableExists(
        EVALUATION_TABLE
    ):

        assert_table_contract(
            EVALUATION_TABLE,
            EVALUATION_CONTRACT
        )


    # ========================================================
    # 24. PERSISTENCIA
    # ========================================================

    if not spark.catalog.tableExists(
        EVALUATION_TABLE
    ):

        (
            evaluations

            .write

            .format(
                "delta"
            )

            .mode(
                "overwrite"
            )

            .option(
                "overwriteSchema",
                "true"
            )

            .saveAsTable(
                EVALUATION_TABLE
            )
        )

    else:

        (
            evaluations

            .write

            .format(
                "delta"
            )

            .mode(
                "append"
            )

            .saveAsTable(
                EVALUATION_TABLE
            )
        )


    print(
        f"✔ Evaluaciones persistidas: "
        f"{EVALUATION_TABLE}"
    )


    # ========================================================
    # 25. RECONCILIAR PERSISTENCIA
    # ========================================================

    persisted_evaluations = (

        spark.table(
            EVALUATION_TABLE
        )

        .filter(
            F.col(
                "evaluation_run_id"
            )
            ==
            EVALUATION_RUN_ID
        )
    )


    PERSISTED_COUNT = (
        persisted_evaluations.count()
    )


    if (
        PERSISTED_COUNT
        !=
        EXPECTED_COUNT
    ):

        raise RuntimeError(
            f"""
            ❌ RECONCILIACIÓN DE EVALUACIÓN FALLÓ

            Esperadas:
            {EXPECTED_COUNT}

            Persistidas:
            {PERSISTED_COUNT}
            """
        )


    print(
        f"✔ Evaluaciones reconciliadas: "
        f"{PERSISTED_COUNT:,}"
    )


    # ========================================================
    # 26. FINALIZAR RUN
    # ========================================================

    update_run_status(
        "SUCCESS"
    )


    print(
        "✔ Evaluation Run finalizado: SUCCESS."
    )


except Exception as exc:

    try:

        update_run_status(
            "FAILED"
        )

    except Exception as update_exc:

        print(
            "⚠ No fue posible actualizar "
            "evaluation_run a FAILED."
        )

        print(
            update_exc
        )


    print()
    print(
        "❌ TRUST & SAFETY EVALUATION FAILED"
    )

    print(
        str(
            exc
        )
    )

    raise


# COMMAND ----------

























 # 10 · Gobierno en Unity Catalog

 `slv_review_evaluation` pertenece físicamente a Silver,
 pero funcionalmente corresponde al dominio Trust & Safety.

 Se publica metadata que describe:

 - dominio;
 - producto;
 - clasificación;
 - criticidad;
 - entidad;
 - presencia de identificadores.

 ## Clasificación

 Se utiliza:

 `classification = confidential`

 porque la tabla relaciona identificadores de Review con
 resultados heurísticos y razones de evaluación.

 ## Importante

 Los tags no representan controles de acceso.

 Los controles de autorización mediante `GRANT`,
 vistas seguras u otras políticas se gestionan separadamente.

 `data_owner` y `data_steward` se incorporarán cuando se
 formalice el RACI. No se asignan valores ficticios.


In [0]:

# COMMAND ----------

# ============================================================
# 27. METADATA UNITY CATALOG
# ============================================================

def apply_evaluation_uc_metadata(
    table_name
):

    description = (
        "Evaluaciones heurísticas Trust & Safety por Review, "
        "versión de regla y Evaluation Run. "
        "SIGNAL representa una señal heurística y no constituye "
        "confirmación de fraude."
    )


    safe_description = (
        description.replace(
            "'",
            "''"
        )
    )


    spark.sql(
        f"""
        COMMENT ON TABLE {table_name}
        IS '{safe_description}'
        """
    )


    try:

        spark.sql(
            f"""
            ALTER TABLE {table_name}
            SET TAGS (
                'data_layer' = 'silver',
                'data_domain' = 'trust_safety',
                'data_product' = 'ts_rule_evaluations',
                'source_system' = 'yelp',
                'classification' = 'confidential',
                'criticality' = 'high',
                'entity' = 'review_rule_evaluation',
                'personal_data' = 'identifier'
            )
            """
        )

        print(
            f"✔ Tags UC aplicados: "
            f"{table_name}"
        )

    except Exception as exc:

        print(
            "⚠ La evaluación fue persistida, "
            "pero no fue posible aplicar todos los tags UC."
        )

        print(
            f"Detalle: {exc}"
        )


apply_evaluation_uc_metadata(
    EVALUATION_TABLE
)


# COMMAND ----------


 # 10.1 · Diccionario funcional

Las columnas principales reciben comentarios explícitos para
evitar interpretaciones incorrectas.

En particular:

 `risk_score` en esta tabla es el **aporte de peso de una regla**
 cuando produce `SIGNAL`.

No representa:

 - probabilidad de fraude;
 - score consolidado;
 - decisión final de investigación.


In [0]:
# COMMAND ----------

# ============================================================
# 28. COMMENTS DE COLUMNAS
# ============================================================

COLUMN_COMMENTS = {

    "evaluation_id":
        (
            "Identificador determinístico de la evaluación "
            "Review + Rule Version + Evaluation Run."
        ),

    "review_id":
        (
            "Identificador de la Review evaluada."
        ),

    "rule_id":
        (
            "Identificador lógico de la regla Trust & Safety."
        ),

    "rule_version_id":
        (
            "Versión exacta de la regla utilizada en "
            "la evaluación."
        ),

    "evaluation_result":
        (
            "Resultado heurístico de la regla: SIGNAL, "
            "NO_SIGNAL o NOT_EVALUABLE."
        ),

    "risk_score":
        (
            "Aporte de peso de la regla cuando genera SIGNAL. "
            "No representa una probabilidad de fraude ni el "
            "score consolidado de la Review."
        ),

    "reason":
        (
            "Explicación técnica del resultado de la "
            "evaluación heurística."
        ),

    "evaluation_run_id":
        (
            "Identificador de la ejecución Trust & Safety "
            "que produjo la evaluación."
        ),

    "evaluated_at":
        (
            "Timestamp de ejecución de la evaluación."
        )
}


for (
    column_name,
    description
) in COLUMN_COMMENTS.items():

    safe_description = (
        description.replace(
            "'",
            "''"
        )
    )


    try:

        spark.sql(
            f"""
            ALTER TABLE {EVALUATION_TABLE}
            ALTER COLUMN {column_name}
            COMMENT '{safe_description}'
            """
        )

    except Exception as exc:

        print(
            f"⚠ No fue posible aplicar COMMENT "
            f"a {column_name}: {exc}"
        )


print(
    "✔ Proceso de metadata funcional finalizado."
)


# COMMAND ----------




 # 11 · Resultados por regla

 Se presenta la distribución:

 - `SIGNAL`;
- `NO_SIGNAL`;
 - `NOT_EVALUABLE`.

Estas cantidades corresponden a evaluaciones por regla,
no a Reviews únicas.

In [0]:

# COMMAND ----------

# ============================================================
# 29. RESULTADOS POR REGLA
# ============================================================

persisted_evaluations = (

    spark.table(
        EVALUATION_TABLE
    )

    .filter(
        F.col(
            "evaluation_run_id"
        )
        ==
        EVALUATION_RUN_ID
    )
)


display(

    persisted_evaluations

    .groupBy(
        "rule_id",
        "evaluation_result"
    )

    .count()

    .orderBy(
        "rule_id",
        "evaluation_result"
    )
)


# COMMAND ----------



 # 12 · Tasa de activación por regla

La tasa de activación indica qué porcentaje de las evaluaciones
produjo `SIGNAL`.

Una tasa alta no demuestra fraude.

Debe interpretarse como comportamiento del motor heurístico
sobre el dataset evaluado.


In [0]:
# COMMAND ----------

# ============================================================
# 30. ESTADÍSTICAS POR REGLA
# ============================================================

rule_statistics = (

    persisted_evaluations

    .groupBy(
        "rule_id"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "reviews_evaluated"
        ),

        F.sum(

            F.when(
                F.col(
                    "evaluation_result"
                )
                ==
                "SIGNAL",

                1
            )

            .otherwise(
                0
            )
        ).alias(
            "signals"
        ),

        F.sum(

            F.when(
                F.col(
                    "evaluation_result"
                )
                ==
                "NOT_EVALUABLE",

                1
            )

            .otherwise(
                0
            )
        ).alias(
            "not_evaluable"
        )
    )

    .withColumn(
        "signal_rate_pct",

        F.round(

            F.col(
                "signals"
            )

            /

            F.col(
                "reviews_evaluated"
            )

            *
            100,

            4
        )
    )

    .orderBy(
        "rule_id"
    )
)


display(
    rule_statistics
)


# COMMAND ----------






 # 13 · Señales detectadas

Se muestran hasta 100 evaluaciones con resultado `SIGNAL`
 únicamente como evidencia técnica.

Este resultado:

- no es una lista de fraude;
- no es todavía una cola de investigación;
- no representa una decisión final.

 La consolidación por Review se realiza posteriormente en:

`05_risk_consolidation`.


In [0]:
# COMMAND ----------

# ============================================================
# 31. SEÑALES
# ============================================================

display(

    persisted_evaluations

    .filter(
        F.col(
            "evaluation_result"
        )
        ==
        "SIGNAL"
    )

    .select(

        "review_id",
        "rule_id",
        "rule_version_id",

        "risk_score",
        "reason",

        "evaluation_run_id"
    )

    .orderBy(
        "rule_id",
        "review_id"
    )

    .limit(
        100
    )
)


# COMMAND ----------


# 14 · Resumen de ejecución

Las métricas finales se calculan sobre los registros
efectivamente persistidos para el `evaluation_run_id` actual.

Esto permite reconciliar ejecución y evidencia histórica.


In [0]:
# ============================================================
# 32. RESUMEN FINAL
# ============================================================

rule_summary = (

    persisted_evaluations

    .groupBy(
        "evaluation_result"
    )

    .count()
)


signal_row = (

    rule_summary

    .filter(
        F.col(
            "evaluation_result"
        )
        ==
        "SIGNAL"
    )

    .select(
        "count"
    )

    .first()
)


SIGNAL_EVALUATION_COUNT = (

    signal_row[
        "count"
    ]

    if signal_row

    else 0
)


not_evaluable_row = (

    rule_summary

    .filter(
        F.col(
            "evaluation_result"
        )
        ==
        "NOT_EVALUABLE"
    )

    .select(
        "count"
    )

    .first()
)


NOT_EVALUABLE_COUNT = (

    not_evaluable_row[
        "count"
    ]

    if not_evaluable_row

    else 0
)


no_signal_row = (

    rule_summary

    .filter(
        F.col(
            "evaluation_result"
        )
        ==
        "NO_SIGNAL"
    )

    .select(
        "count"
    )

    .first()
)


NO_SIGNAL_COUNT = (

    no_signal_row[
        "count"
    ]

    if no_signal_row

    else 0
)


# COMMAND ----------

 # Definition of Done — Trust & Safety Evaluation

 - [x] Quality Gate Silver validado.
 - [x] Ejecución DQ Silver completa validada.
 - [x] Contratos de gobierno validados.
 - [x] `evaluation_run` reutilizada sin crear infraestructura.
 - [x] Catálogo gobernado T&S consumido.
 - [x] Solo reglas activas participan.
 - [x] Cada regla activa posee exactamente una versión ACTIVE.
 - [x] Reglas DRAFT no participan.
 - [x] Parámetros provienen de `evaluation_rule_version`.
 - [x] Thresholds provienen de `evaluation_rule_version`.
 - [x] Pesos provienen de `evaluation_rule_version`.
 - [x] Pesos ACTIVE suman 1.0.
 - [x] Features requeridas validadas.
 - [x] Dataset ligado a un único `feature_execution_id`.
 - [x] Grano Review × Rule Version × Run validado.
 - [x] Evaluaciones persistidas y reconciliadas.
 - [x] `evaluation_run` finaliza SUCCESS o FAILED.
 - [x] Metadata Unity Catalog publicada.
 - [x] `risk_score` documentado como aporte de peso.
 - [x] No se consolida riesgo en esta etapa.
 - [x] No se genera una cola de investigación.
 - [x] No se confirma fraude.

 ---

 ## Siguiente etapa

 `05_risk_consolidation`

 La siguiente etapa consolidará las señales por Review y
 construirá el score heurístico ponderado.


In [0]:
# COMMAND ----------

# ============================================================
# 33. RESULTADO PARA ORQUESTADOR
# ============================================================
EVALUATION_FINISHED_AT = (
    datetime.now(
        timezone.utc
    )
)


summary = {

    "status":
        "SUCCESS",

    "pipeline":
        PIPELINE_NAME,

    "pipeline_version":
        PIPELINE_VERSION,

    "evaluation_run_id":
        EVALUATION_RUN_ID,

    "dataset_version":
        DATASET_VERSION,

    "started_at":
        EVALUATION_STARTED_AT
        .isoformat(),

    "finished_at":
        EVALUATION_FINISHED_AT
        .isoformat(),

    "dev_mode":
        DEV_MODE,

    "reviews_evaluated":
        review_count,

    "active_rules":
        ACTIVE_RULE_COUNT,

    "evaluations_expected":
        EXPECTED_COUNT,

    "evaluations_persisted":
        PERSISTED_COUNT,

    "signal_evaluations":
        SIGNAL_EVALUATION_COUNT,

    "no_signal_evaluations":
        NO_SIGNAL_COUNT,

    "not_evaluable":
        NOT_EVALUABLE_COUNT,

    "output_table":
        EVALUATION_TABLE,

    "data_domain":
        "trust_safety",

    "data_product":
        "ts_rule_evaluations"
}


print()
print("=" * 72)
print("TRUST & SAFETY EVALUATION FINALIZADO")
print("=" * 72)

print(
    json.dumps(
        summary,
        indent=2
    )
)

print("=" * 72)


dbutils.notebook.exit(
    json.dumps(
        summary
    )
)